# 04 - Handling the class imbalance

Fraud is about 0.17% of transactions, so a model can learn to say "genuine" every time and still look right almost always. This notebook compares four ways of dealing with that: SMOTE, Borderline-SMOTE, ADASYN and class weighting.

**Split method used for everything here:** walk-forward validation on `development.csv` (5 time-ordered blocks, 4 rounds; see notebook 03). `test.csv` is never loaded in this notebook.

## Part 1: The wrong way, on purpose

**This section is a deliberate demonstration of a mistake. Its results are never used for any decision.**

**SMOTE** creates synthetic fraud examples: it takes a real fraud case, finds its nearest fraud neighbours, and invents new points on the lines between them.

- **Right way:** SMOTE only sees the fraud cases in the round's learning blocks; the check block stays unseen until scoring.
- **Wrong way:** SMOTE is run on the learning blocks **and** the check block together. Some synthetic fraud is then built partly from check-block fraud and goes into training, so the model is shown blurred copies of the very cases it is later scored on.

Everything else is identical between the two runs.

In [1]:
import sys

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

print(sys.executable)

development = pd.read_csv("../data/processed/development.csv")
print(development.shape)

C:\Users\RevathyHarichandran\Desktop\portfolio projects\Fraud_detection\.venv\Scripts\python.exe


(226980, 34)


### Step 1: One round, and the features

**Round 3** is used: it learns from blocks 1 to 3 and is checked on block 4. Its check block holds the most fraud (88 after the gap), so the comparison is clearest.

- `learn`: blocks 1 to 3.
- `check_scored`: block 4 without its gap rows: what the model is scored on.
- `check_all`: all of block 4, gap rows included: what the wrong way feeds into SMOTE ("the check block").

**Features:** V1 to V28 plus Amount, built with the `for` loop from notebook 01, Step 6. **Time is left out:** every check-block Time is later than anything in training, so a model could latch onto "bigger Time" in a way that means nothing real. (Time information is tested properly in Phase 5, as an hour-of-day feature.)

In [2]:
feature_columns = []
for number in range(1, 29):
    feature_columns.append("V" + str(number))
feature_columns.append("Amount")

learn = development[development["block"] <= 3]
check_all = development[development["block"] == 4]
check_scored = development[(development["block"] == 4) & (~development["in_gap"])]

print("learn:", len(learn), "rows,", learn["Class"].sum(), "fraud")
print("check block (all):", len(check_all), "rows,", check_all["Class"].sum(), "fraud")
print("check block (scored):", len(check_scored), "rows,", check_scored["Class"].sum(), "fraud")

learn: 136186 rows, 258 fraud
check block (all): 45398 rows, 89 fraud
check block (scored): 44114 rows, 88 fraud


### Step 2: Scale Amount (the same way in both runs)

Amount runs into the thousands while the V columns sit near zero. A **StandardScaler** turns each value into "how many standard deviations from the mean" (subtract the mean, divide by the standard deviation; its defaults `with_mean=True, with_std=True` do exactly that). It is **fitted on the learning blocks only** and then applied to the check block, the rule for every fitting step. Full explanation in Phase 5.

`scaler.transform(...)` returns a plain numbers table (a numpy array) with one column; `[:, 0]` means "all rows, first column", turning it back into a single column. `.copy()` makes a separate table, so changing Amount here doesn't change the original data.

In [3]:
scaler = StandardScaler()
scaler.fit(learn[["Amount"]])

learn_X = learn[feature_columns].copy()
learn_X["Amount"] = scaler.transform(learn[["Amount"]])[:, 0]
learn_y = learn["Class"]

check_all_X = check_all[feature_columns].copy()
check_all_X["Amount"] = scaler.transform(check_all[["Amount"]])[:, 0]
check_all_y = check_all["Class"]

check_scored_X = check_scored[feature_columns].copy()
check_scored_X["Amount"] = scaler.transform(check_scored[["Amount"]])[:, 0]
check_scored_y = check_scored["Class"]

### Step 3: SMOTE, the right way and the wrong way

SMOTE settings: `k_neighbors=5` (its default: each synthetic point is built from a fraud case and one of its 5 nearest fraud neighbours); `sampling_strategy="auto"` (its default: create enough synthetic fraud to match the number of genuine rows); `random_state=42` (its default is `None`, different results every run; a fixed number makes them reproducible). Other settings are compared properly later, not here.

`fit_resample` returns the original rows first, unchanged and in order, then the new synthetic rows (checked with a small test before writing this). So for the wrong way:

1. join the learning rows and the whole check block with `pd.concat` (stacks tables on top of each other),
2. run SMOTE on the joined table,
3. keep the learning rows plus every synthetic row, and drop the original check rows. `.iloc` picks rows by position: `.iloc[:n]` is the first n rows, `.iloc[n:]` everything from position n on.

One side effect: because the wrong way sees more genuine rows, `"auto"` makes a slightly different number of synthetic rows. Both runs follow the same rule (balance what SMOTE sees), so this is part of doing it the wrong way, not a separate difference.

In [4]:
# Right way: SMOTE sees the learning blocks only.
right_smote = SMOTE(k_neighbors=5, sampling_strategy="auto", random_state=42)
right_X, right_y = right_smote.fit_resample(learn_X, learn_y)

# Wrong way: SMOTE sees the learning blocks and the check block together.
combined_X = pd.concat([learn_X, check_all_X])
combined_y = pd.concat([learn_y, check_all_y])
wrong_smote = SMOTE(k_neighbors=5, sampling_strategy="auto", random_state=42)
resampled_X, resampled_y = wrong_smote.fit_resample(combined_X, combined_y)

number_of_learn_rows = len(learn_X)
number_of_original_rows = len(combined_X)
wrong_X = pd.concat([resampled_X.iloc[:number_of_learn_rows], resampled_X.iloc[number_of_original_rows:]])
wrong_y = pd.concat([resampled_y.iloc[:number_of_learn_rows], resampled_y.iloc[number_of_original_rows:]])

print("right way training rows:", len(right_X), "| fraud (real + synthetic):", right_y.sum())
print("wrong way training rows:", len(wrong_X), "| fraud (real + synthetic):", wrong_y.sum())

right way training rows: 271856 | fraud (real + synthetic): 135928
wrong way training rows: 317076 | fraud (real + synthetic): 181148


### Step 4: Train the same model both ways, score the same check block

**Logistic Regression** is a basic model that turns the features into a fraud-risk score between 0 and 1 (full explanation in Phase 5). `max_iter=1000`: the default of 100 learning steps often isn't enough on data this size and triggers a "did not finish" warning. Other settings are its defaults: `C=1.0` (how strongly the model is held back from over-fitting, explained in Phase 5), solver `lbfgs`, no class weighting.

`predict_proba` gives two columns per transaction, the chance of genuine and the chance of fraud; `[:, 1]` takes the fraud column.

**Score: average precision**, the area under the precision-recall curve across the whole curve. **Precision** is "of the transactions flagged as fraud, how many really are"; **recall** is "of all real fraud, how much was flagged". Average precision summarises precision across every level of recall: 1.0 is perfect, and a model guessing at random scores about the fraud rate (about 0.002 here). The project's headline metric, the same idea restricted to the low-recall range, is explained in Phase 6; the full-curve version is enough to show a leak.

In [5]:
right_model = LogisticRegression(max_iter=1000)
right_model.fit(right_X, right_y)
right_scores = right_model.predict_proba(check_scored_X)[:, 1]

wrong_model = LogisticRegression(max_iter=1000)
wrong_model.fit(wrong_X, wrong_y)
wrong_scores = wrong_model.predict_proba(check_scored_X)[:, 1]

right_ap = average_precision_score(check_scored_y, right_scores)
wrong_ap = average_precision_score(check_scored_y, wrong_scores)
print(f"right way average precision: {right_ap:.4f}")
print(f"wrong way average precision: {wrong_ap:.4f}")
print(f"difference (wrong minus right): {wrong_ap - right_ap:+.4f}")

right way average precision: 0.8402
wrong way average precision: 0.8579
difference (wrong minus right): +0.0177


### Step 5: Is the gap real, or luck?

A 95% bootstrap confidence interval, the same idea as notebook 02, Chart 5 (1,000 resamples, seed 42). Here each resample draws check-block **row positions** with replacement, and both runs are scored on exactly the same resampled rows, so the **difference** between them gets its own interval too. If that interval sits entirely above zero, the wrong way's advantage is very unlikely to be luck.

`rng.integers(0, n, size=n)` draws n whole numbers from 0 up to n - 1, with repeats allowed: row positions resampled with replacement. `.to_numpy()` turns the answers into a plain list of numbers so they can be picked by position.

In [6]:
number_of_resamples = 1000
rng = np.random.default_rng(42)

answers = check_scored_y.to_numpy()
number_of_check_rows = len(answers)

right_aps = []
wrong_aps = []
differences = []
for resample_number in range(number_of_resamples):
    positions = rng.integers(0, number_of_check_rows, size=number_of_check_rows)
    right_ap_resample = average_precision_score(answers[positions], right_scores[positions])
    wrong_ap_resample = average_precision_score(answers[positions], wrong_scores[positions])
    right_aps.append(right_ap_resample)
    wrong_aps.append(wrong_ap_resample)
    differences.append(wrong_ap_resample - right_ap_resample)

print(f"right way: {right_ap:.4f}  95% CI {np.percentile(right_aps, 2.5):.4f} to {np.percentile(right_aps, 97.5):.4f}")
print(f"wrong way: {wrong_ap:.4f}  95% CI {np.percentile(wrong_aps, 2.5):.4f} to {np.percentile(wrong_aps, 97.5):.4f}")
print(f"difference: {wrong_ap - right_ap:+.4f}  95% CI {np.percentile(differences, 2.5):+.4f} to {np.percentile(differences, 97.5):+.4f}")

right way: 0.8402  95% CI 0.7619 to 0.9085
wrong way: 0.8579  95% CI 0.7814 to 0.9203
difference: +0.0177  95% CI +0.0056 to +0.0322


**What this shows** (deliberate demonstration, not used for any decision):
- **The leak makes the model look better without making it better.** Average precision rose from 0.8402 (right way) to 0.8579 (wrong way) on exactly the same check-block transactions, because the wrong way trained on synthetic fraud built partly from the 89 check-block fraud cases.
- **It is real, not luck:** the difference's 95% interval (+0.0056 to +0.0322) sits entirely above zero.
- **It is modest here:** only 89 of the 347 fraud cases SMOTE saw came from the check block. The classic version of the mistake (SMOTE on the whole dataset, then a random split) copies far more test fraud into training; it was not run here, so no number is claimed for it.
- **Comparing the right way:** the two runs' separate intervals overlap heavily (0.7619 to 0.9085 against 0.7814 to 0.9203), yet the paired difference interval is clearly above zero. Scoring both on the same transactions and bootstrapping the difference cancels out their shared ups and downs, a sharper test that the four-way comparison will also use.
- **Side note:** the wrong way also trained on more synthetic rows (181,148 fraud rows against 135,928), because `"auto"` balanced a bigger pool. Both effects come from the same mistake: letting SMOTE see the check block.

## Part 2: The real four-way comparison

All four techniques (SMOTE, Borderline-SMOTE, ADASYN, class weighting) go through all four walk-forward rounds: 16 runs. In every round, the scaler and the technique are fitted **on that round's learning blocks only**; the check block is only ever scored, never fitted on.

### Functions

A **function** is a named, reusable recipe: write the steps once, give them a name, and call the name whenever the steps are needed, handing in whatever changes each time.

```python
def add_tax(price, rate):
    tax = price * rate
    return price + tax
```

`def` starts the recipe, `add_tax` is its name, `price` and `rate` are its inputs (**parameters**), and `return` hands back the answer: `add_tax(100, 0.2)` gives 120.

**Why here:** "scale, rebalance, train, score" is needed 16 times. Copying those lines 16 times would be long, and one small typo in one copy could quietly make one technique look better or worse. With functions the steps exist once, so every technique and round is treated identically. (The alternative, writing everything inline inside a long loop, works but buries the steps and is harder to check.)

### `prepare_round`

The same steps as Part 1, written once: take one round's learning blocks and its scored check block (gap rows excluded), fit the Amount scaler on the learning blocks only, apply it to both, and return four things: learning features, learning answers, check features, check answers. A function can hand back several things at once by listing them after `return`; they are caught in the same order with `a, b, c, d = prepare_round(...)`.

In [7]:
def prepare_round(development, round_number):
    learn = development[development["block"] <= round_number]
    check = development[(development["block"] == round_number + 1) & (~development["in_gap"])]

    scaler = StandardScaler()
    scaler.fit(learn[["Amount"]])

    learn_X = learn[feature_columns].copy()
    learn_X["Amount"] = scaler.transform(learn[["Amount"]])[:, 0]
    check_X = check[feature_columns].copy()
    check_X["Amount"] = scaler.transform(check[["Amount"]])[:, 0]

    return learn_X, learn["Class"], check_X, check["Class"]

### `train_and_score`

Handles one technique for one round, and returns fraud-risk scores for the check block:

- **SMOTE**, **Borderline-SMOTE** or **ADASYN**: fit the technique on the learning data only (`fit_resample`), then train the model on the result.
- **class weighting**: no new rows; train the model directly on the learning data with `class_weight="balanced"`, which makes each fraud case count as much as genuine cases do in total (for the development data's counts, a weight of about 284 for fraud against 0.5 for genuine).

`if` / `elif` / `else` picks one path: `elif` means "otherwise, if...". The final `else` stops with an error (`raise ValueError`) if an unknown technique name is passed in, so a typo can never silently run the wrong thing.

**Settings** (defaults confirmed in the installed imbalanced-learn 0.14.2):
- SMOTE: `k_neighbors=5`, `sampling_strategy="auto"` (balance fraud up to the number of genuine rows).
- Borderline-SMOTE: `k_neighbors=5` (fraud neighbours used to build new points), `m_neighbors=10` (all neighbours checked to decide if a fraud case is borderline), `kind="borderline-1"` (new points built only towards other fraud), `sampling_strategy="auto"`.
- ADASYN: `n_neighbors=5`, `sampling_strategy="auto"`.
- All three: `random_state=42` (their default is `None`, different results every run).
- Model: Logistic Regression, `max_iter=1000`, other settings default (as in Part 1).

**Not tuning these settings is a deliberate choice:** the question is which *approach* works best with sensible standard settings. Tuning four techniques would multiply the runs and, with 250 check fraud cases, risk picking settings that happen to suit these particular cases.

**Why Logistic Regression and not XGBoost here:** comparing techniques with a much stronger model would mix two questions, which fix and which model. Phase 5 compares models separately, using whichever fix wins here.

In [8]:
from imblearn.over_sampling import ADASYN, BorderlineSMOTE


def train_and_score(technique, learn_X, learn_y, check_X):
    if technique == "SMOTE":
        resampler = SMOTE(k_neighbors=5, sampling_strategy="auto", random_state=42)
        train_X, train_y = resampler.fit_resample(learn_X, learn_y)
        model = LogisticRegression(max_iter=1000)
    elif technique == "Borderline-SMOTE":
        resampler = BorderlineSMOTE(k_neighbors=5, m_neighbors=10, kind="borderline-1",
                                    sampling_strategy="auto", random_state=42)
        train_X, train_y = resampler.fit_resample(learn_X, learn_y)
        model = LogisticRegression(max_iter=1000)
    elif technique == "ADASYN":
        resampler = ADASYN(n_neighbors=5, sampling_strategy="auto", random_state=42)
        train_X, train_y = resampler.fit_resample(learn_X, learn_y)
        model = LogisticRegression(max_iter=1000)
    elif technique == "class weighting":
        train_X = learn_X
        train_y = learn_y
        model = LogisticRegression(max_iter=1000, class_weight="balanced")
    else:
        raise ValueError("Unknown technique: " + technique)

    model.fit(train_X, train_y)
    return model.predict_proba(check_X)[:, 1]

### The score: AUCPR over the low-recall range

A fraud team can only review a limited number of alerts, so what matters most is **how precise the model is among its most confident flags**: the low-recall end of the precision-recall curve. So the area under the curve is calculated **only for recall from 0 to 0.2**, then divided by 0.2 so the result still runs from 0 to 1. **Why 0.2:** it is the example cutoff in the survey the literature review relies on; fixing it now, before seeing results, stops it being picked to favour any technique.

How the function works:
1. `precision_recall_curve` (scikit-learn) gives matching lists of precision and recall, one pair per possible flagging cutoff, with recall going from 1 down to 0.
2. Average precision is the sum, over the curve's steps, of (how much recall the step covers) × (the precision there). This is scikit-learn's own definition.
3. To keep only recall up to the limit, each step's recall values are capped at the limit with `np.minimum` (it compares two lists value by value and keeps the smaller each time), so steps above the limit cover nothing and the step crossing it is cut short.
4. Divide by the limit.

numpy does steps 2 and 3 on whole lists at once (`recall[:-1]` is every value except the last, `recall[1:]` every value except the first), which is far faster than a Python loop over up to 176,000 steps, repeated thousands of times in the bootstrap.

**Checking my own code:** with the limit set to 1.0 (the whole curve), it must give exactly scikit-learn's `average_precision_score`. Tested below on Part 1's right-way scores.

In [9]:
from sklearn.metrics import precision_recall_curve

recall_limit = 0.2


def aucpr_low_recall(answers, scores, recall_limit):
    precision, recall, thresholds = precision_recall_curve(answers, scores)
    step_top = np.minimum(recall[:-1], recall_limit)
    step_bottom = np.minimum(recall[1:], recall_limit)
    area = np.sum((step_top - step_bottom) * precision[:-1])
    return area / recall_limit


# Check: with the limit at 1.0 it must equal scikit-learn's average precision.
print("mine, limit 1.0:   ", aucpr_low_recall(check_scored_y, right_scores, 1.0))
print("scikit-learn's AP: ", average_precision_score(check_scored_y, right_scores))
print("low-recall AUCPR (limit 0.2) on the same scores:", aucpr_low_recall(check_scored_y, right_scores, recall_limit))

mine, limit 1.0:    0.8402030554530284
scikit-learn's AP:  0.8402030554530284
low-recall AUCPR (limit 0.2) on the same scores: 1.0


The function matches scikit-learn exactly with the limit at 1.0, so it is correct.

But on these scores the low-recall AUCPR is **1.0**: the model's ~18 most confident flags (20% of 88 fraud) were all real fraud. That is good for the model, but if every technique hits this ceiling the headline metric can't tell them apart. **Rule fixed now, before any comparison result exists:** the main test is the paired difference in low-recall AUCPR (recall 0 to 0.2). If none of those paired differences sits clear of zero, the decision falls to **average precision**, judged by the same paired-difference rule.

### Running all 16 combinations

For each round, `prepare_round` is called once, then all four techniques are trained and scored on exactly the same learning and check data. Each technique's check-block scores are kept for pooling, along with the check-block answers. `time.time()` reads the computer's clock in seconds; the difference before and after a run is how long it took. A dictionary (notebook 02) holds one growing list of scores per technique; `np.concatenate` later joins each technique's four lists of scores, one per round, into one long list.

In [10]:
import time

techniques = ["SMOTE", "Borderline-SMOTE", "ADASYN", "class weighting"]

round_results = []
pooled_scores = {}
for technique in techniques:
    pooled_scores[technique] = []
pooled_answers = []

for round_number in range(1, 5):
    learn_X, learn_y, check_X, check_y = prepare_round(development, round_number)
    pooled_answers.append(check_y.to_numpy())

    for technique in techniques:
        start = time.time()
        scores = train_and_score(technique, learn_X, learn_y, check_X)
        seconds = time.time() - start

        pooled_scores[technique].append(scores)
        low_recall = aucpr_low_recall(check_y, scores, recall_limit)
        average_precision = average_precision_score(check_y, scores)
        round_results.append({"round": round_number, "technique": technique,
                              "low_recall_aucpr": low_recall, "average_precision": average_precision,
                              "seconds": seconds})
        print(f"round {round_number}  {technique:17s} low-recall AUCPR {low_recall:.4f}  average precision {average_precision:.4f}  ({seconds:.1f} s)")

round 1  SMOTE             low-recall AUCPR 0.8268  average precision 0.5709  (0.4 s)


round 1  Borderline-SMOTE  low-recall AUCPR 1.0000  average precision 0.5084  (0.3 s)


round 1  ADASYN            low-recall AUCPR 0.6707  average precision 0.4540  (0.5 s)
round 1  class weighting   low-recall AUCPR 0.8292  average precision 0.5896  (0.2 s)


round 2  SMOTE             low-recall AUCPR 0.8637  average precision 0.5848  (1.0 s)


round 2  Borderline-SMOTE  low-recall AUCPR 0.9980  average precision 0.5826  (0.7 s)


...

round 3  class weighting   low-recall AUCPR 1.0000  average precision 0.8413  (0.5 s)


round 4  SMOTE             low-recall AUCPR 1.0000  average precision 0.7817  (1.4 s)


round 4  Borderline-SMOTE  low-recall AUCPR 1.0000  average precision 0.5357  (1.3 s)


round 4  ADASYN            low-recall AUCPR 1.0000  average precision 0.7637  (1.7 s)


round 4  class weighting   low-recall AUCPR 1.0000  average precision 0.7821  (0.5 s)


### Round by round, side by side

`pd.DataFrame(round_results)` turns the list of dictionaries into a table, one row per run. `pivot` rearranges it so each technique is a row and each round a column (`index` = what becomes the rows, `columns` = what becomes the columns, `values` = what fills the cells), so the four techniques can be compared across rounds at a glance. Both tables are saved to `outputs/tables`.

In [11]:
round_table = pd.DataFrame(round_results)
round_table.to_csv("../outputs/tables/imbalance_by_round.csv", index=False)

low_recall_by_round = round_table.pivot(index="technique", columns="round", values="low_recall_aucpr")
average_precision_by_round = round_table.pivot(index="technique", columns="round", values="average_precision")

print("Low-recall AUCPR (recall 0 to 0.2), by round:")
print(low_recall_by_round.round(4))
print()
print("Average precision, by round:")
print(average_precision_by_round.round(4))

Low-recall AUCPR (recall 0 to 0.2), by round:
round                  1       2       3    4
technique                                    
ADASYN            0.6707  0.6882  0.9189  1.0
Borderline-SMOTE  1.0000  0.9980  1.0000  1.0
SMOTE             0.8268  0.8637  1.0000  1.0
class weighting   0.8292  0.8854  1.0000  1.0

Average precision, by round:
round                  1       2       3       4
technique                                       
ADASYN            0.4540  0.5144  0.7861  0.7637
Borderline-SMOTE  0.5084  0.5826  0.7253  0.5357
SMOTE             0.5709  0.5848  0.8402  0.7817
class weighting   0.5896  0.5823  0.8413  0.7821


### Pooled across all four check blocks, with bootstrap intervals

Each technique's check-block scores from the four rounds are joined into one list (250 fraud cases), and scored once. Next to that, the **average of the four round scores**: pooling puts four different round models on one ranking, and their scores may be pitched slightly differently, so if the two tell different stories that is reported.

**Bootstrap** (same as Part 1, Step 5): 1,000 resamples of pooled row positions, seed 42. In each resample all four techniques are scored on exactly the same rows, so each technique gets its own 95% interval, and every **pair** gets a paired-difference interval. The 6 pairs come from a nested loop (notebook 02, Chart 6) over the technique list, each pair once. A difference whose interval sits entirely above or below zero is a real difference; one that includes zero is not proven.

In [12]:
all_answers = np.concatenate(pooled_answers)
all_scores = {}
for technique in techniques:
    all_scores[technique] = np.concatenate(pooled_scores[technique])

print("pooled check rows:", len(all_answers), "| fraud:", all_answers.sum())
print()

# Bootstrap: every technique scored on the same resampled rows each time.
number_of_resamples = 1000
rng = np.random.default_rng(42)
number_of_pooled_rows = len(all_answers)

resampled_low_recall = {}
resampled_average_precision = {}
for technique in techniques:
    resampled_low_recall[technique] = []
    resampled_average_precision[technique] = []

for resample_number in range(number_of_resamples):
    positions = rng.integers(0, number_of_pooled_rows, size=number_of_pooled_rows)
    for technique in techniques:
        resampled_answers = all_answers[positions]
        resampled_scores = all_scores[technique][positions]
        resampled_low_recall[technique].append(aucpr_low_recall(resampled_answers, resampled_scores, recall_limit))
        resampled_average_precision[technique].append(average_precision_score(resampled_answers, resampled_scores))

summary_rows = []
for technique in techniques:
    technique_rounds = round_table[round_table["technique"] == technique]
    summary_rows.append({
        "technique": technique,
        "pooled_low_recall_aucpr": aucpr_low_recall(all_answers, all_scores[technique], recall_limit),
        "low_recall_ci_lower": np.percentile(resampled_low_recall[technique], 2.5),
        "low_recall_ci_upper": np.percentile(resampled_low_recall[technique], 97.5),
        "mean_of_rounds_low_recall": technique_rounds["low_recall_aucpr"].mean(),
        "pooled_average_precision": average_precision_score(all_answers, all_scores[technique]),
        "ap_ci_lower": np.percentile(resampled_average_precision[technique], 2.5),
        "ap_ci_upper": np.percentile(resampled_average_precision[technique], 97.5),
        "mean_of_rounds_ap": technique_rounds["average_precision"].mean(),
    })

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv("../outputs/tables/imbalance_pooled.csv", index=False)
print(summary_table.round(4).T)

pooled check rows: 176175 | fraud: 250



                                0                 1       2                3
technique                   SMOTE  Borderline-SMOTE  ADASYN  class weighting
pooled_low_recall_aucpr    0.9975               1.0  0.9053           0.9966
low_recall_ci_lower        0.9864            0.9935  0.8431           0.9831
low_recall_ci_upper           1.0               1.0  0.9604              1.0
mean_of_rounds_low_recall  0.9226            0.9995  0.8194           0.9287
pooled_average_precision   0.6861            0.5687  0.6223           0.6923
ap_ci_lower                0.6248            0.5046  0.5528            0.631
ap_ci_upper                0.7423            0.6324  0.6883           0.7492
mean_of_rounds_ap          0.6944             0.588  0.6295           0.6988


In [13]:
pair_rows = []
for first in range(len(techniques)):
    for second in range(first + 1, len(techniques)):
        name_a = techniques[first]
        name_b = techniques[second]

        low_recall_differences = []
        ap_differences = []
        for resample_number in range(number_of_resamples):
            low_recall_differences.append(resampled_low_recall[name_a][resample_number] - resampled_low_recall[name_b][resample_number])
            ap_differences.append(resampled_average_precision[name_a][resample_number] - resampled_average_precision[name_b][resample_number])

        pair_rows.append({
            "pair (A minus B)": name_a + " minus " + name_b,
            "low_recall_difference": summary_table.loc[first, "pooled_low_recall_aucpr"] - summary_table.loc[second, "pooled_low_recall_aucpr"],
            "low_recall_ci_lower": np.percentile(low_recall_differences, 2.5),
            "low_recall_ci_upper": np.percentile(low_recall_differences, 97.5),
            "ap_difference": summary_table.loc[first, "pooled_average_precision"] - summary_table.loc[second, "pooled_average_precision"],
            "ap_ci_lower": np.percentile(ap_differences, 2.5),
            "ap_ci_upper": np.percentile(ap_differences, 97.5),
        })

pair_table = pd.DataFrame(pair_rows)
pair_table.to_csv("../outputs/tables/imbalance_paired_differences.csv", index=False)
print(pair_table.round(4).to_string(index=False))

                      pair (A minus B)  low_recall_difference  low_recall_ci_lower  low_recall_ci_upper  ap_difference  ap_ci_lower  ap_ci_upper
          SMOTE minus Borderline-SMOTE                -0.0025              -0.0127               0.0029         0.1175       0.0816       0.1537
                    SMOTE minus ADASYN                 0.0923               0.0366               0.1516         0.0638       0.0440       0.0864
           SMOTE minus class weighting                 0.0009              -0.0000               0.0040        -0.0062      -0.0127       0.0014
         Borderline-SMOTE minus ADASYN                 0.0947               0.0385               0.1566        -0.0536      -0.0973      -0.0066
Borderline-SMOTE minus class weighting                 0.0034              -0.0029               0.0155        -0.1236      -0.1617      -0.0839
          ADASYN minus class weighting                -0.0914              -0.1506              -0.0357        -0.0700      -0.092

**What this shows** (results only; the verdict is the next step):
- **The headline metric hits its ceiling** in rounds 3 and 4 (almost every technique scores 1.0 there); only rounds 1 and 2 separate techniques on it.
- **ADASYN is clearly worse on the headline metric:** all three of its low-recall paired differences sit clear of zero (for example SMOTE minus ADASYN +0.0923, 95% CI +0.0366 to +0.1516).
- **SMOTE, Borderline-SMOTE and class weighting can't be separated on the headline metric:** all three of their paired differences include zero.
- **On average precision (the pre-declared tie-breaker), Borderline-SMOTE is clearly worse** than SMOTE (−0.1175) and class weighting (−0.1236), while **SMOTE and class weighting are tied** (SMOTE minus class weighting −0.0062, 95% CI −0.0127 to +0.0014).
- **Borderline-SMOTE behaves differently:** best at the very top of the ranking in rounds 1 and 2 (low-recall 1.0000 and 0.9980), worst over the whole curve in rounds 1 and 4.
- **Pooled and mean-of-rounds disagree on the headline metric:** averaged round by round, Borderline-SMOTE looks best (0.9995 against about 0.92); pooled, the top three are tied. Pooling ranks four round models' scores together, and those scores are pitched differently, so the strongest rounds dominate the top of the pooled ranking.
- **The literature's expectation did not hold here:** neither Borderline-SMOTE nor ADASYN beat plain SMOTE overall.

## Verdict

**No single technique clearly beat all the others.**

- The pre-declared tie-breaker is applied **by its intent**: the top three were tied on the headline metric, so average precision decides among them.
- **Ranking:** SMOTE = class weighting, then Borderline-SMOTE, then ADASYN. ADASYN is clearly worse on the headline metric; Borderline-SMOTE is clearly worse on average precision; **SMOTE and class weighting are tied on both metrics.**
- **The answer changes over time:** Borderline-SMOTE was sharpest at the top in rounds 1 and 2, the headline metric hit its ceiling in rounds 3 and 4, and SMOTE and class weighting traded places on average precision by tiny margins.
- **Against the literature's expectation,** neither Borderline-SMOTE nor ADASYN beat plain SMOTE here.

**Carried forward to Phase 5: class weighting.** This is a practical tie-break between two techniques the data could not separate, **not a performance win**: it invents no data, keeps training fast, carries straight over to XGBoost, and is the cost-sensitive approach the literature review named.